# exp-003 conservative-numeric-regularized — rolling back capacity after a real-LB regression

In [ ]:
import os
os.environ["PYTHONHASHSEED"] = "42"
import json
import random
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import cloudpickle
import lightgbm as lgb
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import KFold


In [ ]:
PROJECT_ROOT = "../../.."
EXP_DIR = "."
SEED = 42
N_SPLITS = 5
N_THREADS = 4
TFIDF_MAX_FEATURES = 6000
TFIDF_MIN_DF = 3
SVD_COMPONENTS = 60
N_ESTIMATORS = 500
LEARNING_RATE = 0.05
NUM_LEAVES = 31
REG_ALPHA = 1.0
REG_LAMBDA = 1.0
MIN_CHILD_SAMPLES = 30
FEATURE_FRACTION = 0.8
BAGGING_FRACTION = 0.8
BAGGING_SEED = 42


In [ ]:
random.seed(SEED)
np.random.seed(SEED)

sys.path.insert(0, str(Path(PROJECT_ROOT).resolve()))
from tools.config import load_config, resolve_metric

root = Path(PROJECT_ROOT).resolve()
exp_dir = Path(EXP_DIR).resolve()
cfg = load_config(root)
mae = resolve_metric(cfg["task"]["metric"], root)


## Load

In [ ]:
raw = root / cfg["paths"]["raw"]
train = pd.read_csv(raw / "train.csv")
test = pd.read_csv(raw / "test.csv")
print(train.shape, test.shape)


## Feature engineering: numeric-magnitude regex (kept) + small TF-IDF/SVD (reverted)

The numeric-magnitude regex extraction from exp-002 is kept unchanged -- it captures verified real signal (`bathroom_count` corr 0.52 with `log1p(price)`) unrelated to any particular row's identity, so it is not implicated in exp-002's real-leaderboard regression. `TFIDF_MAX_FEATURES`/`SVD_COMPONENTS` are reverted to exp-001's smaller, less overfitting-prone values (6,000/60, down from exp-002's 20,000/120).

In [ ]:
PATTERNS_NUM = {
    "bed": (re.compile(r"(\d+)\s*(?:-|\s)?(?:bed(?:room)?s?)\b", re.I), 0, 20),
    "bath": (re.compile(r"(\d+(?:\.\d+)?)\s*(?:-|\s)?(?:full\s+|half\s+)?bath(?:room)?s?\b", re.I), 0, 15),
    "sqft": (re.compile(r"([\d,]{3,7})\s*(?:sq\.?\s?ft|square\s?feet|sqft)", re.I), 100, 20000),
    "acre": (re.compile(r"(\d+(?:\.\d+)?)\s*acres?", re.I), 0, 200),
    "garage": (re.compile(r"(\d+)[\s-]*car\s+garage", re.I), 0, 6),
    "year_built": (re.compile(r"built\s+in\s+(\d{4})", re.I), 1750, 2026),
}
FLAG_PATTERNS = {
    "hoa": r"\bHOA\b",
    "renovated": r"\brenovat\w*\b",
    "new_construction": r"\bnew construction\b",
    "redacted": r"\[Redacted Entity\]",
    "pool": r"\bpool\b",
}


def numeric_regex_features(texts):
    texts = texts.fillna("")
    out = {}
    for name, (pat, lo, hi) in PATTERNS_NUM.items():
        vals = []
        for t in texts:
            m = pat.search(t)
            if m:
                try:
                    v = float(m.group(1).replace(",", ""))
                    if not (lo <= v <= hi):
                        v = np.nan
                except ValueError:
                    v = np.nan
            else:
                v = np.nan
            vals.append(v)
        out[f"num_{name}"] = vals
    for name, pat in FLAG_PATTERNS.items():
        out[f"has_{name}"] = texts.str.contains(pat, regex=True, case=False, na=False).astype(np.float64)
    out["char_len"] = texts.str.len().astype(np.float64)
    out["word_count"] = texts.str.split().str.len().astype(np.float64)
    return pd.DataFrame(out)


## Model: `TextPriceModel` (regularized)

`reg_alpha`/`reg_lambda` (L1/L2 leaf-weight penalties) and `min_child_samples=30` (up from LightGBM's default 20) discourage splits that only fit a handful of training rows. `feature_fraction=0.8`/`bagging_fraction=0.8` (with `bagging_freq=1` and a fixed `bagging_seed` for determinism) mean every tree sees a different random subset of features/rows, which prevents the ensemble from consistently keying off the same corpus-specific quirk across all 500 trees -- the opposite of exp-002's `feature_fraction=bagging_fraction=1.0`, which exposed every tree to the exact same signal, including whatever training-set-specific noise exp-002's real-LB regression showed the model was exploiting.

In [ ]:
class TextPriceModel:
    def __init__(self, seed, n_threads, tfidf_max_features, tfidf_min_df,
                 svd_components, n_estimators, learning_rate, num_leaves,
                 reg_alpha, reg_lambda, min_child_samples,
                 feature_fraction, bagging_fraction, bagging_seed):
        self.seed = seed
        self.n_threads = n_threads
        self.tfidf_max_features = tfidf_max_features
        self.tfidf_min_df = tfidf_min_df
        self.svd_components = svd_components
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.num_leaves = num_leaves
        self.reg_alpha = reg_alpha
        self.reg_lambda = reg_lambda
        self.min_child_samples = min_child_samples
        self.feature_fraction = feature_fraction
        self.bagging_fraction = bagging_fraction
        self.bagging_seed = bagging_seed

    def _feature_matrix(self, texts, fit):
        texts = texts.fillna("")
        if fit:
            self.word_tfidf = TfidfVectorizer(
                ngram_range=(1, 2), max_features=self.tfidf_max_features,
                min_df=self.tfidf_min_df, sublinear_tf=True)
            self.char_tfidf = TfidfVectorizer(
                analyzer="char_wb", ngram_range=(3, 5),
                max_features=self.tfidf_max_features,
                min_df=self.tfidf_min_df, sublinear_tf=True)
            word_sparse = self.word_tfidf.fit_transform(texts)
            char_sparse = self.char_tfidf.fit_transform(texts)
            self.word_svd = TruncatedSVD(n_components=self.svd_components, random_state=self.seed)
            self.char_svd = TruncatedSVD(n_components=self.svd_components, random_state=self.seed)
            word_dense = self.word_svd.fit_transform(word_sparse)
            char_dense = self.char_svd.fit_transform(char_sparse)
        else:
            word_dense = self.word_svd.transform(self.word_tfidf.transform(texts))
            char_dense = self.char_svd.transform(self.char_tfidf.transform(texts))
        return np.hstack([word_dense, char_dense, numeric_regex_features(texts).values])

    def fit(self, texts, y):
        X = self._feature_matrix(texts, fit=True)
        target = np.log1p(y)
        self.model = lgb.LGBMRegressor(
            objective="regression_l1",
            n_estimators=self.n_estimators,
            learning_rate=self.learning_rate,
            num_leaves=self.num_leaves,
            reg_alpha=self.reg_alpha,
            reg_lambda=self.reg_lambda,
            min_child_samples=self.min_child_samples,
            feature_fraction=self.feature_fraction,
            feature_fraction_seed=self.seed,
            bagging_fraction=self.bagging_fraction,
            bagging_freq=1,
            bagging_seed=self.bagging_seed,
            random_state=self.seed,
            deterministic=True,
            force_row_wise=True,
            num_threads=self.n_threads,
            verbose=-1,
        )
        self.model.fit(X, target)
        return self

    def predict_texts(self, texts):
        X = self._feature_matrix(texts, fit=False)
        return np.clip(np.expm1(self.model.predict(X)), 0, None)

    def predict(self, **frames):
        test_df = frames["test"]
        preds = self.predict_texts(test_df["text"])
        return pd.DataFrame({"id": test_df["id"], "listPrice": preds})


def make_model():
    return TextPriceModel(
        seed=SEED, n_threads=N_THREADS,
        tfidf_max_features=TFIDF_MAX_FEATURES, tfidf_min_df=TFIDF_MIN_DF,
        svd_components=SVD_COMPONENTS, n_estimators=N_ESTIMATORS,
        learning_rate=LEARNING_RATE, num_leaves=NUM_LEAVES,
        reg_alpha=REG_ALPHA, reg_lambda=REG_LAMBDA, min_child_samples=MIN_CHILD_SAMPLES,
        feature_fraction=FEATURE_FRACTION, bagging_fraction=BAGGING_FRACTION,
        bagging_seed=BAGGING_SEED,
    )


## Cross-validation

5-fold `KFold`, TF-IDF/SVD refit per fold. Reported for lineage continuity -- per the hypothesis, this CV number is not the basis for the promotion decision.

In [ ]:
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
y = train["listPrice"].values
texts = train["text"]

fold_scores = []
for fold, (tr_idx, va_idx) in enumerate(kf.split(texts)):
    fold_model = make_model()
    fold_model.fit(texts.iloc[tr_idx], y[tr_idx])
    fold_pred = fold_model.predict_texts(texts.iloc[va_idx])
    fold_mae = mae(y[va_idx], fold_pred)
    fold_scores.append(fold_mae)
    print(f"fold {fold}: mae={fold_mae:,.2f}")

cv_primary = float(np.mean(fold_scores))
cv_std = float(np.std(fold_scores))
print(f"\ncv_primary={cv_primary:,.2f}  cv_std={cv_std:,.2f}")


## Save & Load Model

In [ ]:
final_model = make_model()
final_model.fit(texts, y)

with open(exp_dir / "model.pkl", "wb") as fh:
    cloudpickle.dump(final_model, fh)


In [ ]:
with open(exp_dir / "model.pkl", "rb") as fh:
    loaded = cloudpickle.load(fh)


## Prediction

In [ ]:
pred = final_model.predict(test=test)
pred.to_csv(exp_dir / "submission.csv", index=False,
            float_format="%.6f", lineterminator="\n")

pred_loaded = loaded.predict(test=test)
pred_loaded.to_csv(exp_dir / "submission_check.csv", index=False,
                    float_format="%.6f", lineterminator="\n")
assert (pred["listPrice"] == pred_loaded["listPrice"]).all()

metrics_out = {
    "cv_primary": cv_primary,
    "cv_std": cv_std,
    "fold_scores": fold_scores,
    "parent_cv_primary": 363814.4262500411,
    "exp001_lb_public_approx": 460000,
    "exp002_lb_public": 468497.07,
}
print(json.dumps(metrics_out, indent=2))
with open(exp_dir / "cv_metrics.json", "w") as fh:
    json.dump(metrics_out, fh, indent=2)
